19.2 — Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import soundfile as sf
import time


PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)

RESULTS_DIR = PROJECT_DIR / "risultati"

EMBEDDING_DIR = (
    RESULTS_DIR
    / "embedding_multimodali_turn_level"
)

W2V_DIR = (
    RESULTS_DIR
    / "embedding_wav2vec2_turn_level"
)

W2V_DIR.mkdir(
    parents=True,
    exist_ok=True
)


METADATA_PATH = (
    EMBEDDING_DIR
    / "metadata_turni_multimodali.csv"
)


metadata_df = pd.read_csv(
    METADATA_PATH
)


print(
    "Turni:",
    len(metadata_df)
)

print(
    "Pazienti:",
    metadata_df["patient_id"].nunique()
)

print(
    "Durata mediana:",
    round(
        metadata_df[
            "turn_duration_seconds"
        ].median(),
        3
    ),
    "s"
)

print(
    "Durata massima:",
    round(
        metadata_df[
            "turn_duration_seconds"
        ].max(),
        3
    ),
    "s"
)

Turni: 3710
Pazienti: 90
Durata mediana: 1.738 s
Durata massima: 57.881 s


19.3 — Caricamento Wav2Vec2

In [ ]:
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2Model
)


W2V_MODEL_NAME = "facebook/wav2vec2-base"

W2V_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    W2V_DEVICE
)


start = time.time()


w2v_feature_extractor = (
    AutoFeatureExtractor.from_pretrained(
        W2V_MODEL_NAME
    )
)


w2v_model = (
    Wav2Vec2Model.from_pretrained(
        W2V_MODEL_NAME
    )
    .to(W2V_DEVICE)
)


w2v_model.eval()


print(
    "Modello:",
    W2V_MODEL_NAME
)

print(
    "Hidden size:",
    w2v_model.config.hidden_size
)

print(
    "Tempo caricamento:",
    round(
        time.time() - start,
        1
    ),
    "secondi"
)

Device: cpu


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

c:\Users\acer\Desktop\ProgettoTesi\.venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--facebook--wav2vec2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

...

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modello: facebook/wav2vec2-base
Hidden size: 768
Tempo caricamento: 69.3 secondi


model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

19.4 — Funzioni Wav2Vec2

In [3]:
W2V_SR = 16000


def load_audio_w2v(path):

    audio, sr = sf.read(
        str(path),
        dtype="float32"
    )

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != W2V_SR:
        raise ValueError(
            f"Sample rate inatteso: {sr}"
        )

    return audio


def wav2vec2_embedding_batch(
    audio_list
):

    inputs = w2v_feature_extractor(
        audio_list,
        sampling_rate=W2V_SR,
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )


    input_values = (
        inputs["input_values"]
        .to(W2V_DEVICE)
    )

    attention_mask = (
        inputs["attention_mask"]
        .to(W2V_DEVICE)
    )


    with torch.inference_mode():

        output = w2v_model(
            input_values=input_values,
            attention_mask=attention_mask,
            return_dict=True
        )

        hidden = (
            output.last_hidden_state
        )


    # Numero di frame realmente validi
    input_lengths = (
        attention_mask.sum(
            dim=1
        )
    )

    output_lengths = (
        w2v_model
        ._get_feat_extract_output_lengths(
            input_lengths
        )
        .to(torch.long)
    )


    embeddings = []

    for i in range(
        hidden.shape[0]
    ):

        valid_frames = int(
            output_lengths[i].item()
        )

        emb = (
            hidden[
                i,
                :valid_frames,
                :
            ]
            .mean(dim=0)
            .cpu()
            .numpy()
        )

        embeddings.append(
            emb
        )


    return np.vstack(
        embeddings
    )

19.5 — Smoke test

In [4]:
test_indices = (
    metadata_df.sample(
        n=4,
        random_state=42
    ).index
)


test_audio = [
    load_audio_w2v(
        metadata_df.loc[
            i,
            "audio_path_turn"
        ]
    )
    for i in test_indices
]


start = time.time()


test_embeddings = (
    wav2vec2_embedding_batch(
        test_audio
    )
)


elapsed = (
    time.time() - start
)


print(
    "Shape:",
    test_embeddings.shape
)

print(
    "NaN:",
    np.isnan(
        test_embeddings
    ).sum()
)

print(
    "Inf:",
    np.isinf(
        test_embeddings
    ).sum()
)

print(
    "Tempo 4 turni:",
    round(
        elapsed,
        2
    ),
    "secondi"
)

Shape: (4, 768)
NaN: 0
Inf: 0
Tempo 4 turni: 2.13 secondi


19.6 — Benchmark 20 turni

In [ ]:
benchmark_indices = (
    metadata_df.sample(
        n=20,
        random_state=123
    ).index
    .to_numpy()
)


BATCH_SIZE = 4

benchmark_embeddings = []

start = time.time()


for start_idx in range(
    0,
    len(benchmark_indices),
    BATCH_SIZE
):

    batch_indices = (
        benchmark_indices[
            start_idx:
            start_idx + BATCH_SIZE
        ]
    )

    batch_audio = [
        load_audio_w2v(
            metadata_df.loc[
                i,
                "audio_path_turn"
            ]
        )
        for i in batch_indices
    ]

    batch_emb = (
        wav2vec2_embedding_batch(
            batch_audio
        )
    )

    benchmark_embeddings.append(
        batch_emb
    )


benchmark_embeddings = np.vstack(
    benchmark_embeddings
)


elapsed = (
    time.time() - start
)


seconds_per_turn = (
    elapsed
    / len(benchmark_indices)
)


estimated_minutes = (
    seconds_per_turn
    * len(metadata_df)
    / 60
)


print(
    "Shape:",
    benchmark_embeddings.shape
)

print(
    "Tempo totale:",
    round(elapsed, 2),
    "secondi"
)

print(
    "Secondi per turno:",
    round(
        seconds_per_turn,
        3
    )
)

print(
    "Stima 3710 turni:",
    round(
        estimated_minutes,
        1
    ),
    "minuti"
)

print(
    "Stima ore:",
    round(
        estimated_minutes / 60,
        2
    )
)

Shape: (20, 768)
Tempo totale: 5.07 secondi
Secondi per turno: 0.254
Stima 3710 turni: 15.7 minuti
Stima ore: 0.26


19.7 — Estrazione completa Wav2Vec2 con checkpoint

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import time


W2V_EMBEDDING_PATH = (
    W2V_DIR
    / "embedding_audio_wav2vec2_base_turn_level.npy"
)

W2V_PROGRESS_PATH = (
    W2V_DIR
    / "embedding_audio_wav2vec2_progress.csv"
)


N_TURNS = len(metadata_df)
EMBED_DIM = w2v_model.config.hidden_size

BATCH_SIZE = 4
CHECKPOINT_EVERY = 100

MAX_SECONDS = 30.0
MAX_SAMPLES = int(
    MAX_SECONDS * W2V_SR
)


# ------------------------------------------------------------
# Resume
# ------------------------------------------------------------

if W2V_EMBEDDING_PATH.exists():

    w2v_embeddings = np.load(
        W2V_EMBEDDING_PATH
    )

    if w2v_embeddings.shape != (
        N_TURNS,
        EMBED_DIM
    ):
        raise ValueError(
            "Shape checkpoint inattesa: "
            f"{w2v_embeddings.shape}"
        )

    print("Checkpoint trovato.")

else:

    w2v_embeddings = np.full(
        (
            N_TURNS,
            EMBED_DIM
        ),
        np.nan,
        dtype=np.float32
    )

    print("Nuova estrazione.")


completed = (
    np.isfinite(
        w2v_embeddings
    ).all(axis=1)
)


print(
    "Già completati:",
    int(completed.sum()),
    "/",
    N_TURNS
)

Nuova estrazione.
Già completati: 0 / 3710


19.8 — Gestione audio > 30 secondi

In [ ]:
def wav2vec2_embedding_long_audio(
    audio
):

    chunks = [
        audio[start:start + MAX_SAMPLES]
        for start in range(
            0,
            len(audio),
            MAX_SAMPLES
        )
    ]

    chunk_embeddings = []
    chunk_weights = []

    for chunk in chunks:

        emb = wav2vec2_embedding_batch(
            [chunk]
        )[0]

        chunk_embeddings.append(
            emb
        )

        chunk_weights.append(
            len(chunk)
        )


    return np.average(
        np.vstack(
            chunk_embeddings
        ),
        axis=0,
        weights=np.asarray(
            chunk_weights
        )
    ).astype(np.float32)

19.9 — Estrazione completa

In [8]:
start_total = time.time()

pending_indices = np.where(
    ~completed
)[0]


# Separiamo corti e lunghi
short_indices = [
    i
    for i in pending_indices
    if metadata_df.loc[
        i,
        "turn_duration_seconds"
    ] <= 30
]

long_indices = [
    i
    for i in pending_indices
    if metadata_df.loc[
        i,
        "turn_duration_seconds"
    ] > 30
]


# Ordiniamo i corti per durata:
# riduce il padding all'interno dei batch
short_indices = sorted(
    short_indices,
    key=lambda i:
        metadata_df.loc[
            i,
            "turn_duration_seconds"
        ]
)


print(
    "Turni <=30 s:",
    len(short_indices)
)

print(
    "Turni >30 s:",
    len(long_indices)
)


processed_since_save = 0


with tqdm(
    total=N_TURNS,
    initial=int(completed.sum()),
    desc="Wav2Vec2 turn-level"
) as pbar:


    # --------------------------------------------------------
    # Turni <= 30 secondi
    # --------------------------------------------------------

    for start_idx in range(
        0,
        len(short_indices),
        BATCH_SIZE
    ):

        batch_indices = (
            short_indices[
                start_idx:
                start_idx + BATCH_SIZE
            ]
        )

        batch_audio = [
            load_audio_w2v(
                metadata_df.loc[
                    i,
                    "audio_path_turn"
                ]
            )
            for i in batch_indices
        ]


        batch_embeddings = (
            wav2vec2_embedding_batch(
                batch_audio
            )
        )


        for i, emb in zip(
            batch_indices,
            batch_embeddings
        ):

            w2v_embeddings[i] = (
                emb.astype(
                    np.float32
                )
            )


        processed_since_save += (
            len(batch_indices)
        )

        pbar.update(
            len(batch_indices)
        )


        # ----------------------------------------------------
        # Checkpoint
        # ----------------------------------------------------

        if (
            processed_since_save
            >= CHECKPOINT_EVERY
        ):

            np.save(
                W2V_EMBEDDING_PATH,
                w2v_embeddings
            )

            completed_now = (
                np.isfinite(
                    w2v_embeddings
                ).all(axis=1)
            )

            pd.DataFrame(
                {
                    "turn_id":
                        metadata_df[
                            "turn_id"
                        ],

                    "completed":
                        completed_now
                }
            ).to_csv(
                W2V_PROGRESS_PATH,
                index=False
            )

            processed_since_save = 0


    # --------------------------------------------------------
    # Turni > 30 secondi
    # --------------------------------------------------------

    for i in long_indices:

        audio = load_audio_w2v(
            metadata_df.loc[
                i,
                "audio_path_turn"
            ]
        )

        w2v_embeddings[i] = (
            wav2vec2_embedding_long_audio(
                audio
            )
        )

        pbar.update(1)


# ------------------------------------------------------------
# Salvataggio finale
# ------------------------------------------------------------

np.save(
    W2V_EMBEDDING_PATH,
    w2v_embeddings
)


completed_final = (
    np.isfinite(
        w2v_embeddings
    ).all(axis=1)
)


pd.DataFrame(
    {
        "turn_id":
            metadata_df["turn_id"],

        "completed":
            completed_final
    }
).to_csv(
    W2V_PROGRESS_PATH,
    index=False
)


elapsed_total = (
    time.time()
    - start_total
)


print("\nEstrazione terminata.")

print(
    "Completati:",
    int(completed_final.sum()),
    "/",
    N_TURNS
)

print(
    "Tempo totale:",
    round(
        elapsed_total / 60,
        1
    ),
    "minuti"
)

print(
    "File:",
    W2V_EMBEDDING_PATH
)

Turni <=30 s: 3704
Turni >30 s: 6


Wav2Vec2 turn-level:   0%|          | 0/3710 [00:00<?, ?it/s]


Estrazione terminata.
Completati: 3710 / 3710
Tempo totale: 22.8 minuti
File: C:\Users\acer\Desktop\ProgettoTesi\risultati\embedding_wav2vec2_turn_level\embedding_audio_wav2vec2_base_turn_level.npy


19.10 — QC embedding Wav2Vec2

In [9]:
w2v_embeddings_final = np.load(
    W2V_EMBEDDING_PATH
)


print(
    "Shape:",
    w2v_embeddings_final.shape
)

print(
    "NaN:",
    int(
        np.isnan(
            w2v_embeddings_final
        ).sum()
    )
)

print(
    "Inf:",
    int(
        np.isinf(
            w2v_embeddings_final
        ).sum()
    )
)

print(
    "Turni:",
    len(metadata_df)
)

print(
    "Pazienti:",
    metadata_df[
        "patient_id"
    ].nunique()
)

Shape: (3710, 768)
NaN: 0
Inf: 0
Turni: 3710
Pazienti: 90


19.11 — Normalizzazione e campione bilanciato

In [ ]:
from sklearn.preprocessing import normalize


# ------------------------------------------------------------
# Normalizzazione L2
# ------------------------------------------------------------

X_w2v = normalize(
    w2v_embeddings_final,
    norm="l2"
).astype(np.float32)


print(
    "Norma media:",
    np.linalg.norm(
        X_w2v,
        axis=1
    ).mean()
)


# ------------------------------------------------------------
# Stesso criterio del Notebook 18
# massimo 25 turni per paziente
# ------------------------------------------------------------

MAX_TURNS_PER_PATIENT = 25
RANDOM_STATE = 42


rng = np.random.default_rng(
    RANDOM_STATE
)

selected_indices = []


for patient_id, group in metadata_df.groupby(
    "patient_id",
    sort=False
):

    indices = group.index.to_numpy()

    if len(indices) > MAX_TURNS_PER_PATIENT:

        indices = rng.choice(
            indices,
            size=MAX_TURNS_PER_PATIENT,
            replace=False
        )

    selected_indices.extend(
        indices
    )


balanced_indices_w2v = np.sort(
    np.asarray(
        selected_indices
    )
)


balanced_metadata_w2v = (
    metadata_df
    .loc[
        balanced_indices_w2v
    ]
    .copy()
)


X_w2v_bal = X_w2v[
    balanced_indices_w2v
]


print(
    "Turni bilanciati:",
    len(
        balanced_indices_w2v
    )
)

print(
    "Pazienti:",
    balanced_metadata_w2v[
        "patient_id"
    ].nunique()
)

Norma media: 1.0
Turni bilanciati: 2156
Pazienti: 90


19.12 — PCA Wav2Vec2

In [11]:
# ============================================================
# 19.12 — PCA Wav2Vec2
# ============================================================

from sklearn.decomposition import PCA


pca_w2v = PCA(
    n_components=0.95,
    svd_solver="full"
)


X_w2v_pca = pca_w2v.fit_transform(
    X_w2v_bal
)


print(
    "Dimensioni originali:",
    X_w2v_bal.shape[1]
)

print(
    "Componenti PCA:",
    X_w2v_pca.shape[1]
)

print(
    "Varianza conservata:",
    round(
        pca_w2v.explained_variance_ratio_.sum(),
        4
    )
)

Dimensioni originali: 768
Componenti PCA: 114
Varianza conservata: 0.9501


19.13 — KMeans Wav2Vec2

In [12]:
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


w2v_kmeans_results = []
w2v_kmeans_models = {}
w2v_kmeans_labels = {}


for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(
        X_w2v_pca
    )


    counts = pd.Series(
        labels
    ).value_counts()


    row = {
        "k": k,

        "silhouette":
            silhouette_score(
                X_w2v_pca,
                labels
            ),

        "calinski_harabasz":
            calinski_harabasz_score(
                X_w2v_pca,
                labels
            ),

        "davies_bouldin":
            davies_bouldin_score(
                X_w2v_pca,
                labels
            ),

        "smallest_cluster_n":
            int(
                counts.min()
            ),

        "largest_cluster_n":
            int(
                counts.max()
            )
    }


    w2v_kmeans_results.append(
        row
    )

    w2v_kmeans_models[k] = model
    w2v_kmeans_labels[k] = labels


    print(
        f"K={k} | "
        f"sil={row['silhouette']:.4f} | "
        f"CH={row['calinski_harabasz']:.2f} | "
        f"DB={row['davies_bouldin']:.4f} | "
        f"min={row['smallest_cluster_n']} | "
        f"max={row['largest_cluster_n']}"
    )


w2v_kmeans_results_df = pd.DataFrame(
    w2v_kmeans_results
)


display(
    w2v_kmeans_results_df.round(4)
)

K=2 | sil=0.2053 | CH=642.55 | DB=1.7904 | min=1002 | max=1154
K=3 | sil=0.1222 | CH=430.32 | DB=2.2921 | min=654 | max=832
K=4 | sil=0.1172 | CH=343.04 | DB=2.3052 | min=342 | max=772
K=5 | sil=0.1011 | CH=292.34 | DB=2.4615 | min=308 | max=585
K=6 | sil=0.0867 | CH=255.19 | DB=2.4875 | min=291 | max=435


,k,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,2,0.2053,642.5501,1.7904,1002,1154
1,3,0.1222,430.3203,2.2921,654,832
2,4,0.1172,343.0363,2.3052,342,772
3,5,0.1011,292.3445,2.4615,308,585
4,6,0.0867,255.1860,2.4875,291,435


19.14 — Audit durata del miglior clustering Wav2Vec2

In [13]:
from scipy.stats import (
    kruskal,
    pointbiserialr
)


BEST_K_W2V = int(
    w2v_kmeans_results_df
    .sort_values(
        "silhouette",
        ascending=False
    )
    .iloc[0]["k"]
)


print(
    "Miglior K per silhouette:",
    BEST_K_W2V
)


# ------------------------------------------------------------
# Assegnazione di tutti i 3710 turni
# ------------------------------------------------------------

X_w2v_all_pca = (
    pca_w2v.transform(
        X_w2v
    )
)


best_w2v_model = (
    w2v_kmeans_models[
        BEST_K_W2V
    ]
)


w2v_labels_all = (
    best_w2v_model.predict(
        X_w2v_all_pca
    )
)


w2v_audit_df = (
    metadata_df.copy()
)


w2v_audit_df[
    "wav2vec2_cluster"
] = w2v_labels_all


# ------------------------------------------------------------
# Durata per cluster
# ------------------------------------------------------------

duration_summary_w2v = (
    w2v_audit_df
    .groupby(
        "wav2vec2_cluster"
    )
    .agg(
        n_turns=(
            "turn_id",
            "count"
        ),

        duration_mean=(
            "turn_duration_seconds",
            "mean"
        ),

        duration_median=(
            "turn_duration_seconds",
            "median"
        ),

        duration_std=(
            "turn_duration_seconds",
            "std"
        )
    )
    .reset_index()
)


display(
    duration_summary_w2v.round(3)
)


# ------------------------------------------------------------
# Test globale della durata tra cluster
# ------------------------------------------------------------

duration_groups = [
    group[
        "turn_duration_seconds"
    ].to_numpy()

    for _, group
    in w2v_audit_df.groupby(
        "wav2vec2_cluster"
    )
]


kw_stat, kw_p = kruskal(
    *duration_groups
)


print(
    "\nKruskal-Wallis durata:"
)

print(
    "H =",
    round(
        kw_stat,
        3
    ),
    "| p =",
    f"{kw_p:.3e}"
)

Miglior K per silhouette: 2


,wav2vec2_cluster,n_turns,duration_mean,duration_median,duration_std
0,0,1657,3.646,1.721,5.002
1,1,2053,2.460,1.755,2.302



Kruskal-Wallis durata:
H = 1.406 | p = 2.357e-01


19.15 — Distribuzione cluster per quartile di durata

In [15]:
w2v_audit_df[
    "duration_quartile"
] = pd.qcut(
    w2v_audit_df[
        "turn_duration_seconds"
    ],
    q=4,
    labels=[
        "Q1_shortest",
        "Q2",
        "Q3",
        "Q4_longest"
    ],
    duplicates="drop"
)


duration_cluster_w2v = pd.crosstab(
    w2v_audit_df[
        "duration_quartile"
    ],
    w2v_audit_df[
        "wav2vec2_cluster"
    ],
    normalize="index"
)


display(
    duration_cluster_w2v.round(3)
)

wav2vec2_cluster,0,1
duration_quartile,,
Q1_shortest,0.493,0.507
Q2,0.412,0.588
Q3,0.356,0.644
Q4_longest,0.525,0.475


19.16 — Campione bilanciato identico al Notebook 18

In [ ]:
MAX_TURNS_PER_PATIENT = 25
RANDOM_STATE = 42


balanced_indices_reference = (
    metadata_df
    .groupby(
        "patient_id",
        group_keys=False
    )
    .apply(
        lambda group:
            group.sample(
                n=min(
                    len(group),
                    MAX_TURNS_PER_PATIENT
                ),
                random_state=RANDOM_STATE
            )
    )
    .index
    .to_numpy()
)


balanced_indices_reference = np.sort(
    balanced_indices_reference
)


print(
    "Turni campione di riferimento:",
    len(balanced_indices_reference)
)

print(
    "Pazienti:",
    metadata_df
    .loc[balanced_indices_reference, "patient_id"]
    .nunique()
)


same_indices = np.array_equal(
    balanced_indices_w2v,
    balanced_indices_reference
)


print(
    "Il campione usato finora era già identico:",
    same_indices
)

Turni campione di riferimento: 2156
Pazienti: 90
Il campione usato finora era già identico: False


C:\Users\acer\AppData\Local\Temp\ipykernel_2468\309732408.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


19.17 — PCA + KMeans sul campione identico a Notebook 18
- ESEGUIRE SOLO SE same_indices == False

In [17]:
balanced_indices_w2v = (
    balanced_indices_reference.copy()
)

balanced_metadata_w2v = (
    metadata_df
    .loc[balanced_indices_w2v]
    .copy()
)

X_w2v_bal = X_w2v[
    balanced_indices_w2v
]


pca_w2v = PCA(
    n_components=0.95,
    svd_solver="full"
)

X_w2v_pca = pca_w2v.fit_transform(
    X_w2v_bal
)


print(
    "PCA:",
    X_w2v_bal.shape[1],
    "->",
    X_w2v_pca.shape[1],
    "| varianza:",
    round(
        pca_w2v.explained_variance_ratio_.sum(),
        4
    )
)


w2v_kmeans_results = []
w2v_kmeans_models = {}
w2v_kmeans_labels = {}


for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(
        X_w2v_pca
    )

    counts = pd.Series(
        labels
    ).value_counts()

    row = {
        "k": k,

        "silhouette":
            silhouette_score(
                X_w2v_pca,
                labels
            ),

        "calinski_harabasz":
            calinski_harabasz_score(
                X_w2v_pca,
                labels
            ),

        "davies_bouldin":
            davies_bouldin_score(
                X_w2v_pca,
                labels
            ),

        "smallest_cluster_n":
            int(counts.min()),

        "largest_cluster_n":
            int(counts.max())
    }

    w2v_kmeans_results.append(row)

    w2v_kmeans_models[k] = model
    w2v_kmeans_labels[k] = labels

    print(
        f"K={k} | "
        f"sil={row['silhouette']:.4f} | "
        f"CH={row['calinski_harabasz']:.2f} | "
        f"DB={row['davies_bouldin']:.4f} | "
        f"min={row['smallest_cluster_n']} | "
        f"max={row['largest_cluster_n']}"
    )


w2v_kmeans_results_df = pd.DataFrame(
    w2v_kmeans_results
)

PCA: 768 -> 114 | varianza: 0.9501
K=2 | sil=0.2048 | CH=639.64 | DB=1.7942 | min=1016 | max=1140
K=3 | sil=0.1216 | CH=429.21 | DB=2.2938 | min=659 | max=821
K=4 | sil=0.1079 | CH=343.15 | DB=2.4404 | min=393 | max=690
K=5 | sil=0.1045 | CH=291.40 | DB=2.4300 | min=357 | max=580
K=6 | sil=0.0867 | CH=255.55 | DB=2.4146 | min=278 | max=440


19.18 — Audit durata definitivo con campione corretto

In [ ]:
BEST_K_W2V = 2

best_w2v_model = w2v_kmeans_models[
    BEST_K_W2V
]


# Tutti i 3710 turni nello spazio PCA corretto
X_w2v_all_pca = pca_w2v.transform(
    X_w2v
)


w2v_labels_all = best_w2v_model.predict(
    X_w2v_all_pca
)


w2v_audit_df = metadata_df.copy()

w2v_audit_df[
    "wav2vec2_cluster"
] = w2v_labels_all


duration_summary_w2v = (
    w2v_audit_df
    .groupby("wav2vec2_cluster")
    .agg(
        n_turns=("turn_id", "count"),
        duration_mean=("turn_duration_seconds", "mean"),
        duration_median=("turn_duration_seconds", "median"),
        duration_std=("turn_duration_seconds", "std")
    )
    .reset_index()
)


display(
    duration_summary_w2v.round(3)
)


from scipy.stats import kruskal

duration_groups = [
    group["turn_duration_seconds"].to_numpy()
    for _, group
    in w2v_audit_df.groupby("wav2vec2_cluster")
]


kw_stat, kw_p = kruskal(
    *duration_groups
)


print(
    "\nKruskal-Wallis durata:"
)

print(
    "H =",
    round(kw_stat, 3),
    "| p =",
    f"{kw_p:.3e}"
)


w2v_audit_df[
    "duration_quartile"
] = pd.qcut(
    w2v_audit_df["turn_duration_seconds"],
    q=4,
    labels=[
        "Q1_shortest",
        "Q2",
        "Q3",
        "Q4_longest"
    ],
    duplicates="drop"
)


duration_cluster_w2v = pd.crosstab(
    w2v_audit_df["duration_quartile"],
    w2v_audit_df["wav2vec2_cluster"],
    normalize="index"
)


display(
    duration_cluster_w2v.round(3)
)

,wav2vec2_cluster,n_turns,duration_mean,duration_median,duration_std
0,0,1683,3.625,1.705,4.975
1,1,2027,2.462,1.772,2.301



Kruskal-Wallis durata:
H = 0.78 | p = 3.771e-01


wav2vec2_cluster,0,1
duration_quartile,,
Q1_shortest,0.505,0.495
Q2,0.419,0.581
Q3,0.357,0.643
Q4_longest,0.532,0.468


19.19 — GMM Wav2Vec2

In [19]:
from sklearn.mixture import GaussianMixture


w2v_gmm_results = []
w2v_gmm_models = {}
w2v_gmm_labels = {}


for k in range(2, 7):

    model = GaussianMixture(
        n_components=k,
        covariance_type="diag",
        random_state=42,
        n_init=3,
        max_iter=500,
        reg_covar=1e-6
    )

    labels = model.fit_predict(
        X_w2v_pca
    )

    probabilities = model.predict_proba(
        X_w2v_pca
    )

    counts = pd.Series(
        labels
    ).value_counts()


    row = {
        "k": k,

        "silhouette":
            silhouette_score(
                X_w2v_pca,
                labels
            ),

        "calinski_harabasz":
            calinski_harabasz_score(
                X_w2v_pca,
                labels
            ),

        "davies_bouldin":
            davies_bouldin_score(
                X_w2v_pca,
                labels
            ),

        "bic":
            model.bic(
                X_w2v_pca
            ),

        "aic":
            model.aic(
                X_w2v_pca
            ),

        "mean_max_probability":
            probabilities.max(
                axis=1
            ).mean(),

        "smallest_cluster_n":
            int(counts.min()),

        "largest_cluster_n":
            int(counts.max()),

        "converged":
            model.converged_
    }


    w2v_gmm_results.append(row)

    w2v_gmm_models[k] = model
    w2v_gmm_labels[k] = labels


    print(
        f"K={k} | "
        f"sil={row['silhouette']:.4f} | "
        f"CH={row['calinski_harabasz']:.2f} | "
        f"DB={row['davies_bouldin']:.4f} | "
        f"BIC={row['bic']:.1f} | "
        f"Pmax={row['mean_max_probability']:.3f}"
    )


w2v_gmm_results_df = pd.DataFrame(
    w2v_gmm_results
)

K=2 | sil=0.0856 | CH=47.75 | DB=5.9938 | BIC=-1131672.3 | Pmax=0.975
K=3 | sil=0.0154 | CH=43.43 | DB=5.9623 | BIC=-1133525.4 | Pmax=0.946
K=4 | sil=0.0278 | CH=148.40 | DB=3.5596 | BIC=-1132838.5 | Pmax=0.939
K=5 | sil=0.0204 | CH=145.58 | DB=4.1476 | BIC=-1132484.0 | Pmax=0.938
K=6 | sil=-0.0049 | CH=106.90 | DB=4.1293 | BIC=-1132920.7 | Pmax=0.938


19.20 — Agglomerative Ward Wav2Vec2

In [20]:
from sklearn.cluster import AgglomerativeClustering


w2v_ward_results = []
w2v_ward_labels = {}


for k in range(2, 7):

    model = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    labels = model.fit_predict(
        X_w2v_pca
    )

    counts = pd.Series(
        labels
    ).value_counts()


    row = {
        "k": k,

        "silhouette":
            silhouette_score(
                X_w2v_pca,
                labels
            ),

        "calinski_harabasz":
            calinski_harabasz_score(
                X_w2v_pca,
                labels
            ),

        "davies_bouldin":
            davies_bouldin_score(
                X_w2v_pca,
                labels
            ),

        "smallest_cluster_n":
            int(counts.min()),

        "largest_cluster_n":
            int(counts.max())
    }


    w2v_ward_results.append(row)

    w2v_ward_labels[k] = labels


    print(
        f"K={k} | "
        f"sil={row['silhouette']:.4f} | "
        f"CH={row['calinski_harabasz']:.2f} | "
        f"DB={row['davies_bouldin']:.4f} | "
        f"min={row['smallest_cluster_n']} | "
        f"max={row['largest_cluster_n']}"
    )


w2v_ward_results_df = pd.DataFrame(
    w2v_ward_results
)

K=2 | sil=0.1882 | CH=567.34 | DB=1.9018 | min=1030 | max=1126
K=3 | sil=0.1128 | CH=357.46 | DB=2.5976 | min=381 | max=1126
K=4 | sil=0.0603 | CH=284.78 | DB=2.8527 | min=381 | max=716
K=5 | sil=0.0620 | CH=246.50 | DB=2.7001 | min=233 | max=716
K=6 | sil=0.0660 | CH=215.20 | DB=2.5613 | min=34 | max=716


19.21 — HDBSCAN Wav2Vec2, PCA 95%

In [ ]:
import hdbscan

w2v_hdbscan_results = []
w2v_hdbscan_labels = {}

for min_cluster_size in [30, 50, 100]:

    model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=None,
        metric="euclidean",
        cluster_selection_method="eom"
    )

    labels = model.fit_predict(
        X_w2v_pca
    )

    non_noise = labels != -1

    n_clusters = len(
        set(labels) - {-1}
    )

    n_noise = int(
        (labels == -1).sum()
    )

    noise_fraction = (
        n_noise / len(labels)
    )

    if (
        n_clusters >= 2
        and
        non_noise.sum() > n_clusters
    ):

        X_valid = X_w2v_pca[
            non_noise
        ]

        labels_valid = labels[
            non_noise
        ]

        sil = silhouette_score(
            X_valid,
            labels_valid
        )

        ch = calinski_harabasz_score(
            X_valid,
            labels_valid
        )

        db = davies_bouldin_score(
            X_valid,
            labels_valid
        )

        counts = pd.Series(
            labels_valid
        ).value_counts()

        smallest = int(
            counts.min()
        )

        largest = int(
            counts.max()
        )

    else:

        sil = np.nan
        ch = np.nan
        db = np.nan
        smallest = np.nan
        largest = np.nan

    row = {
        "min_cluster_size":
            min_cluster_size,

        "n_clusters":
            n_clusters,

        "n_noise":
            n_noise,

        "noise_fraction":
            noise_fraction,

        "silhouette":
            sil,

        "calinski_harabasz":
            ch,

        "davies_bouldin":
            db,

        "smallest_cluster_n":
            smallest,

        "largest_cluster_n":
            largest
    }

    w2v_hdbscan_results.append(
        row
    )

    w2v_hdbscan_labels[
        min_cluster_size
    ] = labels

    print(
        f"min_cluster={min_cluster_size} | "
        f"clusters={n_clusters} | "
        f"noise={noise_fraction:.1%} | "
        f"sil={sil:.4f} | "
        f"CH={ch:.2f} | "
        f"DB={db:.4f}"
    )


w2v_hdbscan_results_df = pd.DataFrame(
    w2v_hdbscan_results
)

display(
    w2v_hdbscan_results_df.round(4)
)

min_cluster=30 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan


,min_cluster_size,n_clusters,n_noise,noise_fraction,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,30,0,2156,1.0,NaN,NaN,NaN,NaN,NaN
1,50,0,2156,1.0,NaN,NaN,NaN,NaN,NaN
2,100,0,2156,1.0,NaN,NaN,NaN,NaN,NaN


19.22 — Label di riferimento per stabilità

In [ ]:
reference_turn_ids_w2v = (
    metadata_df
    .loc[
        balanced_indices_w2v,
        "turn_id"
    ]
    .astype(str)
    .to_numpy()
)


reference_labels_w2v = {
    "KMeans_K2":
        pd.Series(
            w2v_kmeans_labels[2],
            index=reference_turn_ids_w2v
        ),

    "Ward_K2":
        pd.Series(
            w2v_ward_labels[2],
            index=reference_turn_ids_w2v
        ),

    "GMM_K3":
        pd.Series(
            w2v_gmm_labels[3],
            index=reference_turn_ids_w2v
        )
}


print(
    "Turni riferimento:",
    len(reference_turn_ids_w2v)
)

Turni riferimento: 2156


19.22b — Funzione per balanced resampling

In [25]:
def balanced_sample_indices(
    metadata,
    max_turns_per_patient=25,
    seed=42
):

    rng = np.random.default_rng(seed)

    selected = []

    for patient_id, group in metadata.groupby(
        "patient_id",
        sort=False
    ):

        indices = group.index.to_numpy()

        if len(indices) > max_turns_per_patient:

            indices = rng.choice(
                indices,
                size=max_turns_per_patient,
                replace=False
            )

        selected.extend(indices)

    return np.sort(
        np.asarray(selected)
    )


# Test rapido
test_indices = balanced_sample_indices(
    metadata_df,
    max_turns_per_patient=25,
    seed=100
)

print(
    "Turni selezionati:",
    len(test_indices)
)

print(
    "Pazienti:",
    metadata_df.loc[
        test_indices,
        "patient_id"
    ].nunique()
)

Turni selezionati: 2156
Pazienti: 90


19.23 — Stabilità Wav2Vec2

In [ ]:
from sklearn.metrics import adjusted_rand_score

w2v_stability_rows = []


for seed in range(100, 120):
    repeat_indices = balanced_sample_indices(
        metadata_df,
        max_turns_per_patient=25,
        seed=seed
    )

    repeat_turn_ids = (
        metadata_df
        .loc[
            repeat_indices,
            "turn_id"
        ]
        .astype(str)
        .to_numpy()
    )


    X_repeat = pca_w2v.transform(
        X_w2v[
            repeat_indices
        ]
    )

    # KMeans K=2
    km = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=50
    )

    labels_km = km.fit_predict(
        X_repeat
    )

    # Ward K=2
    ward = AgglomerativeClustering(
        n_clusters=2,
        linkage="ward"
    )

    labels_ward = ward.fit_predict(
        X_repeat
    )


    # GMM K=3
    gmm = GaussianMixture(
        n_components=3,
        covariance_type="diag",
        random_state=42,
        n_init=3,
        max_iter=500,
        reg_covar=1e-6
    )

    labels_gmm = gmm.fit_predict(
        X_repeat
    )


    current_labels = {
        "KMeans_K2":
            labels_km,

        "Ward_K2":
            labels_ward,

        "GMM_K3":
            labels_gmm
    }


    for config_name, labels in current_labels.items():
        current = pd.Series(
            labels,
            index=repeat_turn_ids
        )

        reference = (
            reference_labels_w2v[
                config_name
            ]
        )

        common_turns = (
            reference.index
            .intersection(
                current.index
            )
        )

        ari = adjusted_rand_score(
            reference.loc[
                common_turns
            ],
            current.loc[
                common_turns
            ]
        )

        sil = silhouette_score(
            X_repeat,
            labels
        )

        w2v_stability_rows.append(
            {
                "seed":
                    seed,

                "configuration":
                    config_name,

                "n_overlap":
                    len(common_turns),

                "ARI":
                    ari,

                "silhouette":
                    sil
            }
        )

w2v_stability_df = pd.DataFrame(
    w2v_stability_rows
)

print(
    "Esperimenti:",
    len(w2v_stability_df)
)

Esperimenti: 60


19.24 — Riepilogo stabilità Wav2Vec2

In [27]:
w2v_stability_summary = (
    w2v_stability_df
    .groupby(
        "configuration"
    )
    .agg(
        ARI_mean=("ARI", "mean"),
        ARI_std=("ARI", "std"),
        ARI_min=("ARI", "min"),
        ARI_max=("ARI", "max"),
        silhouette_mean=("silhouette", "mean"),
        silhouette_std=("silhouette", "std")
    )
    .reset_index()
    .sort_values(
        "ARI_mean",
        ascending=False
    )
)


display(
    w2v_stability_summary.round(4)
)

,configuration,ARI_mean,ARI_std,ARI_min,ARI_max,silhouette_mean,silhouette_std
1,KMeans_K2,0.9764,0.0135,0.9492,0.9916,0.2068,0.0028
0,GMM_K3,0.7151,0.2265,0.2149,0.9125,0.0238,0.0070
2,Ward_K2,0.5334,0.1385,0.2625,0.7230,0.1788,0.0082


19.25 — Cluster Wav2Vec2 finali su tutti i turni

In [28]:
FINAL_W2V_K = 2

final_w2v_model = w2v_kmeans_models[
    FINAL_W2V_K
]

X_w2v_all_pca = pca_w2v.transform(
    X_w2v
)

final_w2v_labels = final_w2v_model.predict(
    X_w2v_all_pca
)

w2v_final_df = metadata_df.copy()

w2v_final_df[
    "wav2vec2_cluster_k2"
] = final_w2v_labels

print(
    "Turni:",
    len(w2v_final_df)
)

print(
    "\nDistribuzione globale:"
)

display(
    w2v_final_df[
        "wav2vec2_cluster_k2"
    ]
    .value_counts()
    .sort_index()
)

Turni: 3710

Distribuzione globale:


wav2vec2_cluster_k2
0    1683
1    2027
Name: count, dtype: int64

19.26 — Proporzioni Wav2Vec2 per paziente

In [29]:
w2v_patient_profiles = (
    w2v_final_df
    .groupby(
        [
            "patient_id",
            "wav2vec2_cluster_k2"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for cluster in [0, 1]:

    if cluster not in w2v_patient_profiles.columns:

        w2v_patient_profiles[
            cluster
        ] = 0

w2v_patient_profiles = (
    w2v_patient_profiles[
        [0, 1]
    ]
)


w2v_patient_profiles.columns = [
    "n_cluster_0",
    "n_cluster_1"
]

w2v_patient_profiles[
    "n_turns"
] = (
    w2v_patient_profiles[
        "n_cluster_0"
    ]
    +
    w2v_patient_profiles[
        "n_cluster_1"
    ]
)

w2v_patient_profiles[
    "prop_cluster_0"
] = (
    w2v_patient_profiles[
        "n_cluster_0"
    ]
    /
    w2v_patient_profiles[
        "n_turns"
    ]
)


w2v_patient_profiles[
    "prop_cluster_1"
] = (
    w2v_patient_profiles[
        "n_cluster_1"
    ]
    /
    w2v_patient_profiles[
        "n_turns"
    ]
)

w2v_patient_profiles[
    "mixed_clusters"
] = (
    (
        w2v_patient_profiles[
            "n_cluster_0"
        ] > 0
    )
    &
    (
        w2v_patient_profiles[
            "n_cluster_1"
        ] > 0
    )
)

w2v_patient_profiles = (
    w2v_patient_profiles
    .reset_index()
)

print(
    "Pazienti:",
    len(w2v_patient_profiles)
)

print(
    "Pazienti con entrambi i cluster:",
    int(
        w2v_patient_profiles[
            "mixed_clusters"
        ].sum()
    )
)

print(
    "\nProporzione cluster 1:"
)

display(
    w2v_patient_profiles[
        "prop_cluster_1"
    ].describe()
)

Pazienti: 90
Pazienti con entrambi i cluster: 89

Proporzione cluster 1:


count    90.000000
mean      0.512264
std       0.214440
min       0.000000
25%       0.385337
50%       0.543907
75%       0.673423
max       0.900000
Name: prop_cluster_1, dtype: float64

19.27 — Distribuzione delle proporzioni per paziente

In [30]:
w2v_patient_profiles[
    "dominant_cluster"
] = np.where(
    w2v_patient_profiles[
        "prop_cluster_1"
    ] >= 0.5,
    1,
    0
)

w2v_patient_profiles[
    "very_mixed"
] = (
    w2v_patient_profiles[
        "prop_cluster_1"
    ]
    .between(
        0.25,
        0.75,
        inclusive="both"
    )
)

w2v_patient_profiles[
    "nearly_pure"
] = (
    (
        w2v_patient_profiles[
            "prop_cluster_1"
        ] <= 0.10
    )
    |
    (
        w2v_patient_profiles[
            "prop_cluster_1"
        ] >= 0.90
    )
)

print(
    "Pazienti con entrambi i cluster:",
    int(
        w2v_patient_profiles[
            "mixed_clusters"
        ].sum()
    )
)

print(
    "Pazienti con proporzioni 25%-75%:",
    int(
        w2v_patient_profiles[
            "very_mixed"
        ].sum()
    )
)

print(
    "Pazienti quasi puri (<=10% o >=90%):",
    int(
        w2v_patient_profiles[
            "nearly_pure"
        ].sum()
    )
)

print(
    "\nCluster dominante:"
)

display(
    w2v_patient_profiles[
        "dominant_cluster"
    ]
    .value_counts()
    .sort_index()
)

Pazienti con entrambi i cluster: 89
Pazienti con proporzioni 25%-75%: 65
Pazienti quasi puri (<=10% o >=90%): 4

Cluster dominante:


dominant_cluster
0    37
1    53
Name: count, dtype: int64

19.28 — HDBSCAN Wav2Vec2: sensitivity PCA-50

In [ ]:
N_COMPONENTS_HDBSCAN = 50

X_w2v_pca50 = X_w2v_pca[
    :,
    :N_COMPONENTS_HDBSCAN
]

print(
    "Varianza conservata PCA-50:",
    round(
        pca_w2v.explained_variance_ratio_[
            :N_COMPONENTS_HDBSCAN
        ].sum(),
        4
    )
)

w2v_hdbscan_pca50_results = []

for min_cluster_size in [
    30,
    50,
    100
]:
    model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom"
    )

    labels = model.fit_predict(
        X_w2v_pca50
    )

    non_noise = (
        labels != -1
    )

    n_clusters = len(
        set(labels) - {-1}
    )

    n_noise = int(
        (labels == -1).sum()
    )

    noise_fraction = (
        n_noise / len(labels)
    )


    if (
        n_clusters >= 2
        and
        non_noise.sum() > n_clusters
    ):

        X_valid = X_w2v_pca50[
            non_noise
        ]

        labels_valid = labels[
            non_noise
        ]

        sil = silhouette_score(
            X_valid,
            labels_valid
        )

        ch = calinski_harabasz_score(
            X_valid,
            labels_valid
        )

        db = davies_bouldin_score(
            X_valid,
            labels_valid
        )

        counts = pd.Series(
            labels_valid
        ).value_counts()

        smallest = int(
            counts.min()
        )

        largest = int(
            counts.max()
        )

    else:

        sil = np.nan
        ch = np.nan
        db = np.nan
        smallest = np.nan
        largest = np.nan

    row = {
        "min_cluster_size":
            min_cluster_size,

        "min_samples":
            5,

        "n_clusters":
            n_clusters,

        "n_noise":
            n_noise,

        "noise_fraction":
            noise_fraction,

        "silhouette":
            sil,

        "calinski_harabasz":
            ch,

        "davies_bouldin":
            db,

        "smallest_cluster_n":
            smallest,

        "largest_cluster_n":
            largest
    }

    w2v_hdbscan_pca50_results.append(
        row
    )

    print(
        f"min_cluster={min_cluster_size} | "
        f"clusters={n_clusters} | "
        f"noise={noise_fraction:.1%} | "
        f"sil={sil:.4f} | "
        f"CH={ch:.2f} | "
        f"DB={db:.4f}"
    )

w2v_hdbscan_pca50_results_df = pd.DataFrame(
    w2v_hdbscan_pca50_results
)

display(
    w2v_hdbscan_pca50_results_df.round(4)
)

Varianza conservata PCA-50: 0.8828
min_cluster=30 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=50 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan
min_cluster=100 | clusters=0 | noise=100.0% | sil=nan | CH=nan | DB=nan


,min_cluster_size,min_samples,n_clusters,n_noise,noise_fraction,silhouette,calinski_harabasz,davies_bouldin,smallest_cluster_n,largest_cluster_n
0,30,5,0,2156,1.0,NaN,NaN,NaN,NaN,NaN
1,50,5,0,2156,1.0,NaN,NaN,NaN,NaN,NaN
2,100,5,0,2156,1.0,NaN,NaN,NaN,NaN,NaN


19.29 — Salvataggio finale

In [33]:
import joblib

w2v_kmeans_results_df.to_csv(
    W2V_DIR / "risultati_kmeans_wav2vec2.csv",
    index=False
)

w2v_gmm_results_df.to_csv(
    W2V_DIR / "risultati_gmm_wav2vec2.csv",
    index=False
)

w2v_ward_results_df.to_csv(
    W2V_DIR / "risultati_ward_wav2vec2.csv",
    index=False
)

w2v_hdbscan_results_df.to_csv(
    W2V_DIR / "risultati_hdbscan_pca95_wav2vec2.csv",
    index=False
)

w2v_hdbscan_pca50_results_df.to_csv(
    W2V_DIR / "risultati_hdbscan_pca50_wav2vec2.csv",
    index=False
)

w2v_stability_df.to_csv(
    W2V_DIR / "stabilita_clustering_wav2vec2.csv",
    index=False
)

w2v_stability_summary.to_csv(
    W2V_DIR / "riepilogo_stabilita_wav2vec2.csv",
    index=False
)

w2v_final_df.to_csv(
    W2V_DIR / "turni_cluster_wav2vec2_kmeans_k2.csv",
    index=False
)

w2v_patient_profiles.to_csv(
    W2V_DIR / "profili_cluster_wav2vec2_per_paziente.csv",
    index=False
)

duration_summary_w2v.to_csv(
    W2V_DIR / "audit_durata_cluster_wav2vec2.csv",
    index=False
)

duration_cluster_w2v.to_csv(
    W2V_DIR / "cluster_per_quartile_durata_wav2vec2.csv"
)

np.save(
    W2V_DIR / "balanced_indices_reference.npy",
    balanced_indices_w2v
)

joblib.dump(
    pca_w2v,
    W2V_DIR / "pca95_wav2vec2.joblib"
)

joblib.dump(
    final_w2v_model,
    W2V_DIR / "kmeans_k2_wav2vec2.joblib"
)

print("✓ Notebook 19 salvato.")
print("Directory:", W2V_DIR)

✓ Notebook 19 salvato.
Directory: C:\Users\acer\Desktop\ProgettoTesi\risultati\embedding_wav2vec2_turn_level
